In [ ]:
!pip install docling

In [2]:
from docling.document_converter import DocumentConverter

In [3]:
from pathlib import Path

folder = Path("/kaggle/input/datasets/stupidhouse/mypdfs")
pdfs_paths = [str(p) for p in folder.glob("*.pdf")]

In [5]:
from pathlib import Path
import re
from docling.document_converter import DocumentConverter

def extract_start_number(path):
    filename = Path(path).name
    match = re.match(r"(\d+)", filename)
    return int(match.group(1)) if match else float("inf")

pdfs_paths = sorted(pdfs_paths, key=extract_start_number)
print(pdfs_paths)

texts = []
converter = DocumentConverter()

for pdf_path in pdfs_paths:
    result = converter.convert(pdf_path)
    texts.append(result.document.export_to_markdown())
    print(pdf_path, " обработан")

['/kaggle/input/datasets/stupidhouse/mypdfs/01_Elicont_100___1_____04.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/02_Elicont_100_1__2____09_07.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/03_Elicont_100_2__3_____.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/04_Elicont_200___1_____22.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/05_Elicont_200_1__2____14_10.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/06_Elicont_200_2__3_____.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/07_ . .  .pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/08___1____.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/09___2_____23_10_25.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/10_______07_03_25.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/11___1_____14_02_25.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/12___2___18_02_25.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/13___3______.pdf', '/kaggle/input/datasets/stupidhouse/mypdfs/14___4______.pdf', '/kaggle/input/datasets/stupid

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/kaggle/input/datasets/stupidhouse/mypdfs/01_Elicont_100___1_____04.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/02_Elicont_100_1__2____09_07.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/03_Elicont_100_2__3_____.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/04_Elicont_200___1_____22.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/05_Elicont_200_1__2____14_10.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/06_Elicont_200_2__3_____.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/07_ . .  .pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/08___1____.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/09___2_____23_10_25.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/10_______07_03_25.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/11___1_____14_02_25.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/12___2___18_02_25.pdf  обработан
/kaggle/input/datasets/stupidhouse/mypdfs/13___3______.pdf 

In [ ]:
import re

In [9]:
import re

all_filtered_chunks = []
img_count = 1

for text in texts:
    chunks = re.split(r'\n##\s+', text)
    filtered_chunks = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        parts = chunk.split('\n', 1)
        if len(parts) == 2:
            header, content = parts

            if header.strip() and content.strip():
                while '<!-- image -->' in content:
                    content = content.replace('<!-- image -->', f'<img_{img_count}>', 1)
                    img_count += 1

                filtered_chunks.append(header + '\n' + content)

    all_filtered_chunks.append(filtered_chunks)

In [ ]:
for filtered_chunks in all_filtered_chunks[:2]:
    for chunk in filtered_chunks:
        print(chunk)
        print('\n=======================================================================\n')

In [10]:
import json

with open("all_filtered_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_filtered_chunks, f, ensure_ascii=False, indent=2)

In [34]:
import random

atomic_chunks = [chunk for filtered_chunks in all_filtered_chunks for chunk in filtered_chunks]

num_chunks = len(atomic_chunks)
print(num_chunks)

rng = random.Random(42)
rng.shuffle(atomic_chunks)

1284


In [16]:
# Подготовка промпта
prompt = (f"""
    Задача:
    Сгенерируй вопросы на основе предоставленных данных. Ты генерируешь вопросы пользователя, которые он потенциально может задать.
    Ответом на заданный вопрос служат предоставленные данные.
    Формулируй вопросы без упоминания источников, данных или контекста.
    Если предоставленных данных мало, например, 1-2 предложения, придумай 2-3 вопроса.
    Если данных много (больше 2 предложений), придумай по 5 вопросов, затрагивая основные аспекты из данных.

    Позитивные вопросы имеют ОДИН четкий ответ в данных, охватывают РАЗНЫЕ аспекты данных.
    """
    )

In [18]:
from pydantic import BaseModel, Field
from typing import List


class GeneratedQuestions(BaseModel):
    questions: List[str] = Field(
        ...,
        description=(
            "Список пользовательских вопросов, которые можно задать по предоставленным данным. "
            "Каждый вопрос должен быть самостоятельным, без упоминания источника, текста, данных или контекста."
        ),
        min_length=2,
        max_length=5
    )

In [32]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_2 = user_secrets.get_secret("API_KEY2")
secret_value_3 = user_secrets.get_secret("API_KEY3")
secret_value_4 = user_secrets.get_secret("API_KEY4")
secret_value_5 = user_secrets.get_secret("API_KEY5")
secret_value_7 = user_secrets.get_secret("API_KEY7")
secret_value_6 = user_secrets.get_secret("API_KEY_6")


In [33]:
from openai import OpenAI


#инициализация клиентов опен роутер
client1 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_2
  )
client2 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_3
  )
client3 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_4
  )
client4 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_5
  )
client5 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_6
  )
client6 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_7
  )

In [46]:
qa_pairs = [] #массив, куда будут заносится чанки + вопросы

In [ ]:
for i, para in enumerate(atomic_chunks[:300]):
    print(f"\n--- Обрабатывается абзац {i + 1} ---")

    if i < 50:
        client = client1
    elif i < 100:
        client = client2
    elif i < 150:
        client = client3
    elif i < 200:
        client = client4
    elif i < 250:
        client = client5
    elif i < 300:
        client = client6
    else:
        print("Нет клиента для этого индекса")
        continue

    try:
        completion = client.chat.completions.create(
            model="openrouter/elephant-alpha",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": para},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "generated_questions",
                    "strict": True,
                    "schema": GeneratedQuestions.model_json_schema(),
                },
            },
        )
    except Exception as e:
        print(f"Ошибка запроса: {e}")
        continue

    if not (completion and completion.choices):
        print("Ошибка: LLM не вернул ответ")
        continue

    content = completion.choices[0].message.content
    print("Ответ от LLM:\n", content[:1200], "...\n")

    try:
        parsed = GeneratedQuestions.model_validate_json(content)

        if not parsed.questions:
            print("Не найдено вопросов для этого абзаца.")
            continue

        for question in parsed.questions:
            qa_pairs.append({
                "Абзац": para,
                "Позитивный вопрос": question.strip(),
            })

    except Exception as e:
        print("Ошибка при разборе structured output:", e)

In [ ]:
import pandas as pd
# Создание датафрейма
df = pd.DataFrame(qa_pairs)
# Настройки для полного отображения
pd.set_option('display.max_rows', None)  # Показать все строки
pd.set_option('display.max_columns', None)  # Показать все столбцы
pd.set_option('display.width', None)  # Автоматическая ширина
pd.set_option('display.max_colwidth', None)  # Полный текст в ячейках
clean_df = df[ #удаление пустых строк из датафрейма
    ~df['Позитивный вопрос'].str.strip().eq("").fillna(False) &
    ~df['Позитивный вопрос'].str.strip().eq("...").fillna(False)
].dropna( #удаление Nan
    subset=['Абзац', 'Позитивный вопрос'],
    how='any'
).copy()

# Проверка результатов
print(f"Было строк: {len(df)}")
print(f"Стало строк: {len(clean_df)}")

display(clean_df)

In [49]:
clean_df.to_csv('qa_dataset.csv', index=False, encoding='utf-8-sig')